# Practice 34 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — New Concept: Vectorization vs `.apply()`

### What is vectorization?

**`.apply()`** loops over rows or values in Python — one at a time. Python loops are slow because of interpreter overhead.

**Vectorized operations** work on the entire array at once using optimized C code under the hood — no Python loop, much faster.

```python
# Slow — Python loop element by element
df['tip_pct'] = df.apply(lambda row: row['tip'] / row['total_bill'], axis=1)

# Fast — vectorized, operates on whole columns at once
df['tip_pct'] = df['tip'] / df['total_bill']
```

Common vectorized alternatives to `.apply()`:

| Instead of `.apply()` doing... | Use... |
|---|---|
| arithmetic between columns | direct operators: `+`, `-`, `/`, `*` |
| simple if/else on one column | `np.where(condition, true_val, false_val)` |
| string operations | `.str.upper()`, `.str.contains()`, `.str.replace()` |
| binning values | `pd.cut()` or `pd.qcut()` |
| math functions | `np.log()`, `np.sqrt()`, `np.abs()` |

**When `.apply()` is still the right choice:**
- Logic that depends on multiple columns with complex conditions (hard to express with `np.where`)
- Custom functions pandas can't vectorize
- Small DataFrames where readability matters more than speed

### Measuring performance with `%timeit`

```python
%timeit df['tip'] / df['total_bill']                              # vectorized
%timeit df.apply(lambda row: row['tip'] / row['total_bill'], axis=1)  # apply
```

`%timeit` runs the expression many times and reports the average. The difference is often 10–100x.

---
### Task 1: tips — compare approaches

Load `sns.load_dataset('tips')`.

1. Add `tip_pct` two ways: once with `.apply()`, once vectorized. Use `%timeit` on both. How much faster is the vectorized version?
2. Add a `size_label` column: `'large'` if `size >= 4`, else `'small'`. Do it with `.apply()` first, then rewrite it using `np.where`.
3. Add a `bill_category` column using `pd.cut` with 3 equal-width bins, labeled `'low'`, `'mid'`, `'high'`. Could you do this with `.apply()`? Why is `pd.cut` better here?
4. Convert the `sex` column to uppercase using `.str.upper()`. Write the equivalent `.apply()` version — notice how much more verbose it is.

In [ ]:
# Your code here


---
## Level 2 — Rewrite Challenge

### Task 2: rewrite these `.apply()` calls

Each snippet below uses `.apply()`. Rewrite each one as a vectorized operation — no `.apply()` allowed.

```python
diamonds = sns.load_dataset('diamonds')

# A — arithmetic
diamonds['ppc'] = diamonds.apply(lambda row: row['price'] / row['carat'], axis=1)

# B — simple if/else
diamonds['expensive'] = diamonds['price'].apply(lambda x: True if x > 5000 else False)

# C — math function
diamonds['log_price'] = diamonds['price'].apply(lambda x: np.log(x))

# D — string operation
diamonds['cut_upper'] = diamonds['cut'].apply(lambda x: str(x).upper())

# E — multi-condition if/else (3 tiers)
diamonds['tier'] = diamonds['price'].apply(
    lambda x: 'premium' if x > 10000 else 'mid' if x > 3000 else 'budget'
)
```

After rewriting all five, use `%timeit` to compare your vectorized version vs the original `.apply()` on snippet **A** and **E**.

In [ ]:
diamonds = sns.load_dataset('diamonds')

# Your rewrites here


---
## Level 3 — When to use which

### Task 3: penguins — choose the right tool

Load `sns.load_dataset('penguins')`. Drop nulls.

For each of the following, decide whether to use vectorization or `.apply()` — and justify your choice in a comment:

1. Add `bill_ratio` (`bill_length_mm / bill_depth_mm`).
2. Add `size_class`: `'large'` if `body_mass_g > 4500` else `'small'`.
3. Add `island_species` — a string combining island and species: e.g. `'Torgersen_Adelie'`.
4. Add `mass_class` — `'heavy'` if `body_mass_g > species mean` AND `flipper_length_mm > species mean`, `'light'` if both are below, `'mixed'` otherwise. (Hint: think carefully about whether this can be vectorized.)

After adding all columns:
- Group by `species` and `mass_class` — what's the mean `bill_ratio` per group? Use `observed=True`.
- Which species has the most `'heavy'` penguins? Use `np.argmax()`.

In [ ]:
# Your code here


---
## Level 4 — Real-world

### Task 4: mpg — pipeline with vectorization

Load `sns.load_dataset('mpg')`. Drop rows where `horsepower` is null.

Write a `.pipe()` chain with 3 functions. This time, **avoid `.apply()` wherever possible** — use vectorized operations throughout:

- One converts `cylinders` (ordered) and `origin` (unordered) to `category`
- One adds `power_to_weight` and `log_weight` (`np.log`) — both vectorized
- One adds an `efficiency` label — `'high'` (mpg > 30), `'mid'` (20–30), `'low'` (< 20) — use `pd.cut` or `np.where`, not `.apply()`

After the chain:
- Which origin has the highest mean `power_to_weight`? Use `np.argmax()`.
- Build a pivot table: mean `log_weight` by `efficiency` × `origin`, `observed=True`.
- Use `.loc` to extract only `'high'` efficiency cars. What fraction are from Japan?

In [ ]:
# Your code here
